# 재학습된 YOLO 모델 테스트

이제 모델을 재학습했으므로, '테스트용 이미지'를 대상으로 모델을 사용할 수 있습니다.

In [ ]:
# 이 실습을 위해 사전구성된 워크벤치 이미지를 사용하지 않았다면, 아래 줄의 주석을 해제하고 실행하여 필요한 패키지들을 설치할 수 있습니다.
# !pip install --no-cache-dir --no-dependencies -r requirements.txt

from ultralytics import YOLO
from PIL import Image
import numpy as np
import cv2
from matplotlib import pyplot as plt

import remote_infer

## NumPy, OpenCV: 이미지 전처리 및 배열 변환
## Matplotlib: 이미지 시각화
## remote_infer: 실습 환경에서 제공되는 사용자 정의 전처리 모듈, 복잡한 코드는 remote_infer.py 파일로 분리해두었으며 내용이 궁금하시면 열어보셔도 좋습니다.

이전 노트북에서는 YOLO 모델을 재학습하는 과정을 살펴보았습니다.  
하지만 재학습은 시간이 많이 걸리고 GPU가 필요하므로, 실습 중에 직접 수행하기는 어렵습니다.

그래서 여러분의 편의를 위해, 사전에 재학습된 모델을 ONNX 형식으로 변환하여 제공하고 있습니다.
(학습을 완료한 `YOLOv8.pt` 모델은 Pytorch 전용 형식으로 저장되어 있습니다. 이를 ONNX 형식으로 변환하면 다양한 모델 런타임에서 실행할 수 있습니다.)

In [ ]:
# 재학습된 모델을 원격 URL에서 다운로드 합니다. 
# task="detect"를 지정하여 이 모델이 객체 감지용임을 명시

model = YOLO("https://rhods-public.s3.amazonaws.com/demo-models/ic-models/accident/accident_detect.onnx", task="detect")

In [ ]:
# 자동차 사고 이미지에 대해 모델 테스트
image_path = "images/carImage3.jpg"  ## carImage3.jpg는 심각한 사고(정확도 91% 이상)로 분류된 이미지입니다. 

## 이미지 스케일 및 전처리 수행
_, scale, original_image = remote_infer.preprocess(image_path)

original_image: np.ndarray = cv2.imread(image_path)
blob = cv2.dnn.blobFromImage(original_image, size=(640, 640), swapRB=False)
blob = np.ascontiguousarray(blob[0].transpose((1,2,0)))
results = model.predict(blob)

In [ ]:
# 결과에서 모든 정보(유형, 바운딩 박스, 확신도)를 추출합니다.

detections = []
result = results[0]
for box in result.boxes:
    class_id = int(box.cls.item())
    score = box.conf.item()
    unscaled_cords = box.xyxy.squeeze().tolist()
    cords = [round(unscaled_cords[0] * scale[1]), round(unscaled_cords[1] * scale[0]), round(unscaled_cords[2] * scale[1]), round(unscaled_cords[3] * scale[0])]
    detection = {
        'class_id': class_id,
        'class_name': result.names[class_id],
        'confidence': score,
        'box': cords,
        'scale': scale}
    detections.append(detection)
    print(detection)
    remote_infer.draw_bounding_box(original_image, class_id, score, cords[0], cords[1], cords[2], cords[3])

In [ ]:
# 이미지 위에 박스를 그리고, 감지된 클래스의 이름과 감지 확신도(모델이 해당 객체를 얼마나 확신하는지를 나타냄)을 함께 표시합니다.

img = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)
fig = plt.gcf()
fig.set_size_inches(6, 3)
plt.axis('off')
plt.imshow(img)

'carImage3.jpg' 이미지를 분석한 결과, 재학습된 YOLO 모델은 해당 이미지에서 91% 확률로 '자동차 사고'를 정확히 감지하였습니다.  
사고 지점에는 바운딩 박스가 그려지고, `severe 0.91`이라는 레이블이 표시됩니다.
(실습용 리소스가 업데이트 되었다면 위의 자동차 사진에 표시되는 실제 숫자는 다를 수도 있습니다)

이제 사고의 심각도를 감지할 수 있는 모델이 준비되었으므로, 예측 함수(predict function)를 만들고 ModelMesh를 이용해 모델을 서빙해보겠습니다.

**이 작업은 실습 가이드 웹페이지로 돌아가서 계속 진행해 주세요.**

**아직은 `04-05-model-serving.ipynb` 노트북을 열지 마세요.**